# Zeeschuimer NDJSON → CSV (item id + post URL)

This tool reads an `.ndjson` file exported from Zeeschuimer and writes a slim `.csv` with just three columns:

- **item_id** — the observation id that Zeeschuimer assigns to each record (matches the `item_id` field in the NDJSON)
- **url** — a link to the post (or, for comments, to the post the comment is on)
- **source_platform** — e.g. `tiktok.com`, `instagram.com`, `youtube.com`

It works with exports from the official Zeeschuimer extension as well as forks, because it only reads the top-level envelope that the extension adds to every record, plus a few well-known fields inside `data`.

### How to use
1. Click the **Play button** (▶️) on the cell below.
2. Pick one or more `.ndjson` files when prompted.
3. A `.csv` will be downloaded for each input file.

In [ ]:
#@title Upload and Convert { display-mode: "form" }

import json
import os
import pandas as pd
from google.colab import files


def extract_url(record):
    """Return a best-effort link to the post for one NDJSON record.

    Works against the official Zeeschuimer extension as well as forks.
    We deliberately avoid falling back to `source_platform_url` for post
    streams: Zeeschuimer sets it to whatever page was open when the item
    was captured, which on TikTok/Instagram can be the feed page or a
    *different* video the user happened to be viewing — that produces
    wrong URLs for most rows.

    Strategy:
      1. If the platform module populated `data.url`, trust it.
      2. Reconstruct from platform-specific fields inside `data`:
         - tiktok.com:  @<author.uniqueId>/video/<data.id>
         - instagram / threads: /reel/ or /p/ from data.code
         - youtube.com (fork's shorts module): data.videoId
      3. Comment streams (`*-comments`): use `source_platform_url` —
         for comments that is always the parent post URL.
      4. Unknown platform: fall back to `source_platform_url`.
    """
    data = record.get('data') or {}
    platform = (record.get('source_platform') or '').lower()

    url = data.get('url')
    if url:
        return url

    if platform == 'tiktok.com':
        author = data.get('author') or {}
        if isinstance(author, dict):
            uniq = author.get('uniqueId') or author.get('unique_id')
            item_id = data.get('id')
            if uniq and item_id:
                return f'https://www.tiktok.com/@{uniq}/video/{item_id}'
        return None

    if 'instagram' in platform or 'threads' in platform:
        code = data.get('code')
        if code:
            segment = 'reel' if data.get('product_type') == 'clips' else 'p'
            host = 'threads.net' if 'threads' in platform else 'www.instagram.com'
            return f'https://{host}/{segment}/{code}/'
        return None

    if platform == 'youtube.com' and data.get('videoId'):
        return f'https://www.youtube.com/shorts/{data["videoId"]}'

    if platform.endswith('-comments'):
        return record.get('source_platform_url')

    return record.get('source_platform_url')


def convert_one(filename, raw_bytes):
    rows = []
    skipped = 0
    for lineno, line in enumerate(raw_bytes.decode('utf-8').splitlines(), start=1):
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError as e:
            skipped += 1
            print(f'  line {lineno}: skipped (invalid JSON: {e})')
            continue
        rows.append({
            'item_id': rec.get('item_id'),
            'url': extract_url(rec),
            'source_platform': rec.get('source_platform'),
        })

    df = pd.DataFrame(rows, columns=['item_id', 'url', 'source_platform'])
    base, _ = os.path.splitext(filename)
    out = f'{base}.ids.csv'
    df.to_csv(out, index=False, encoding='utf-8-sig')

    print(f'  {len(df)} rows written to {out} ({skipped} skipped)')
    missing_url = df['url'].isna().sum()
    if missing_url:
        platforms = df.loc[df['url'].isna(), 'source_platform'].value_counts().to_dict()
        print(f'  note: {missing_url} record(s) had no resolvable URL. By platform: {platforms}')

    return out


def run():
    uploaded = files.upload()
    if not uploaded:
        print('No file uploaded.')
        return
    for filename, raw in uploaded.items():
        print(f'Processing "{filename}"...')
        try:
            out = convert_one(filename, raw)
            files.download(out)
        except Exception as e:
            import traceback
            print(f'Error processing {filename}: {e}')
            traceback.print_exc()


run()

### Notes

**Which ID is this?** It's the top-level `item_id` that Zeeschuimer assigns to every observation in the NDJSON. For posts it's the platform's post id; for comments it's the comment id.

**Why only three columns?** The rest of the NDJSON mirrors the raw response from each platform's internal API — the nesting varies by platform and changes over time. If you need deeper fields, open the NDJSON directly in a notebook and select the paths you care about for your specific platform.

**Platform coverage.** Tested against Zeeschuimer exports for TikTok and Instagram (official extension) plus YouTube Shorts / TikTok-comments / YT-shorts-comments (fork). Other upstream modules (Threads, Twitter/X, 9GAG, Douyin, …) will fall through to `data.url` if the module sets it, otherwise to `source_platform_url`. The `source_platform` column tells you which rows to sanity-check.